In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/extensions.py:151: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/annotated/checkpoints/pre_cell_10.pickle

trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'SEP_COMMA', 'REGRESSOR']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'SEP_COMMA', 'REGRESSOR']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['FASTAI_LEARNING_RATE']
me:  2
trying: ['random']
me:  1
trying: ['df']
me:  6
trying: ['factor']
me:  5
trying: ['np']
me:  1


Declaring variable warnings
Declaring variable random
Declaring variable np
Declaring variable shutil
Declaring variable pd
Declaring variable traceback
Declaring variable SEP_DOLLAR
Declaring variable CONVERT_TO_CAT
Declaring variable VARIABLE_FILES
Declaring variable AUTO_ADJUST_LEARNING_RATE
Declaring variable SEP_DOLLAR
Declaring variable CONVERT_TO_CAT
Declaring variable VARIABLE_FILES
Declaring variable AUTO_ADJUST_LEARNING_RATE
Declaring variable SHUFFLE_DATA
Declaring variable REGRESSOR
Declaring variable SEP_COMMA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SHUFFLE_DATA
Declaring variable REGRESSOR
Declaring variable SEP_COMMA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SHUFFLE_DATA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SEP_COMMA
Declaring variable REGRESSOR
Declaring variable SHUFFLE_DATA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SEP_COMMA
Declaring variable REGRESSOR
Declaring variable SEP_DOLLAR
Declaring variable CONV

In [4]:
%%RecordEvent
%%time
### cell 10 ###

# Cell 10 optimized for cudf
target = ''
target_str = ''
targets = []
# List of all columns once
df_cols = df.columns.to_list()

# Iterate potential targets in reverse (skip first column)
for col in reversed(df_cols[1:]):
    try:
        # Convert candidate target to float64 on GPU and fill missing values
        df[col] = df[col].astype('float64').fillna(0)
        target = col
        target_str = col.replace('/', '-')
    except Exception:
        continue
    print(f'Target Variable: {target}')

    # Ensure PARAM_DIR and config files exist
    os.makedirs(PARAM_DIR, exist_ok=True)
    for fname in ['cats.txt', 'conts.txt', 'cols_to_delete.txt']:
        fpath = f'{PARAM_DIR}/{fname}'
        if not os.path.exists(fpath):
            open(fpath, 'w').close()

    # Drop duplicate rows and optionally shuffle on GPU
    df = df.drop_duplicates()
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # Convert all boolean columns to uint8 in one step
    bool_cols = [c for c, dt in df.dtypes.to_dict().items() if dt == 'bool']
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype('uint8')

    # Drop columns listed in cols_to_delete.txt in bulk
    with open(f'{PARAM_DIR}/cols_to_delete.txt') as f:
        del_cols = [c for c in f.read().splitlines() if c in df.columns]
    if del_cols:
        df = df.drop(columns=del_cols)

    # Fill any remaining NaNs with zero on GPU
    df = df.fillna(0)

    # Auto-detect categorical vs continuous variables via vectorized ops
    nunique = df.nunique()
    counts = df.count()
    ratio = nunique / counts
    cats = ratio[ratio < 0.05].index.to_list()
    conts = ratio[ratio >= 0.05].index.to_list()

    # Exclude target from feature lists
    cats = [c for c in cats if c != target]
    conts = [c for c in conts if c != target]

    # Override with user-provided files if requested
    if VARIABLE_FILES:
        with open(f'{PARAM_DIR}/cats.txt') as f:
            cats = f.read().splitlines()
        with open(f'{PARAM_DIR}/conts.txt') as f:
            conts = f.read().splitlines()

    # Convert remaining continuous vars to float64 on GPU, handle failures
    valid_conts = []
    for var in conts:
        try:
            df[var] = df[var].astype('float64')
            valid_conts.append(var)
        except Exception:
            print(f'Could not convert {var} to float.')
            if CONVERT_TO_CAT:
                cats.append(var)
    conts = valid_conts

    targets.append(target)


Target Variable: Profit_processed


Could not convert City to float.
Target Variable: Discount_processed
Could not convert City to float.
Target Variable: Quantity_processed
Could not convert City to float.
Target Variable: Sales_processed
Could not convert City to float.
Target Variable: Sub-Category_processed
Could not convert City to float.
Target Variable: Category_processed


Could not convert City to float.
Target Variable: Region_processed
Could not convert City to float.
Target Variable: Postal Code_processed
Could not convert City to float.
Target Variable: State_processed
Could not convert City to float.
Target Variable: City_processed
Could not convert City to float.
Target Variable: Country_processed


Could not convert City to float.
Target Variable: Segment_processed
Could not convert City to float.
Target Variable: Ship Mode_processed
Could not convert City to float.
Target Variable: Profit
Could not convert City to float.
Target Variable: Discount
Could not convert City to float.
Target Variable: Quantity


Could not convert City to float.
Target Variable: Sales
Could not convert City to float.
Target Variable: Postal Code
Could not convert City to float.
CPU times: user 1.01 s, sys: 44.2 ms, total: 1.05 s
Wall time: 1.05 s


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10/checkpoints/post_cell_10_try_7.pickle

migration speed (bps): 783504583.1048671
---------------------------
variables to migrate:
target_str 60
SEP_DOLLAR 24
SHUFFLE_DATA 28
valid_conts 120
REGRESSOR 28
target 60
SEP_COMMA 28
np 72
fpath 170
AUTO_ADJUST_LEARNING_RATE 24
PROJECT_NAME 59
CONVERT_TO_CAT 24
VARIABLE_FILES 24
FASTAI_LEARNING_RATE 24
ENABLE_BREAKPOINT 28
conts 120
ratio 48
fname 67
cats 248
shutil 72
bool_cols 56
nunique 48
counts 48
targets 248
del_cols 56
df_cols 264
pd 72
PARAM_DIR 151
var 65
traceback 72
random 72
f 208
warnings 72
df 48
factor 28
col 56
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10/checkpoints/post_cell_10_try_7.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['PARAM_DIR', 'df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['PARAM_DIR', 'df']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['df']
Intermediate variables ['factor']
Future variables ['PARAM_DIR']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['col', 'df']
Intermediate variables []
Future variables ['PARAM_DIR']
Modified dataframes
  df
    Input columns: set()
    Changed columns: set()
    Created columns: {'City_processed', 'Sub-Category_processed', 'Segment_processed', 'Ship Mode_processed', 'Region_processed', 'State_processed', 'Sales_processed', 'Country_processed', 'Quantity_processed', 'Category_processed', 'Profit_processed', 'Discount_processed', 'Postal Code_processed'}
    Deleted columns: set()
======= Cell 4 =======
In

In [7]:
with open(
    "/home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/opt_cell_exec_info_10_try_7.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[10], f)

In [8]:
opt_output = Out.get(4)

In [9]:
df_opt = df
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/annotated/checkpoints/post_cell_10.pickle
assert compare_df(df_opt, df)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

trying: ['AUTO_ADJUST_LEARNING_RATE', 'SEP_DOLLAR', 'VARIABLE_FILES', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'SEP_DOLLAR', 'VARIABLE_FILES', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['focus_cont', 'cont', 'n', 'var']
me:  13
me:  13
me:  13
me:  13
trying: ['focus_cont', 'cont', 'n', 'var']
me:  13
me:  13
me

ValueError: Shape mismatch: (9977, 26) vs (9962, 26)